# Day 038 — Exercise 3: correlation_summary

**What you'll build:** `correlation_summary(df, target_col) -> pd.DataFrame` — compute the Pearson correlation of every numeric column against `target_col`, return a DataFrame with `feature` and `correlation` columns sorted by absolute correlation descending.

**Why it matters:** Correlation reveals which features move together with your target — the starting point of every feature-selection workflow and the fastest way to find surprising (or suspicious) relationships in data.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# SALES_DF: 8 rows, 6 columns
# product:  Widget×4, Gadget×2, Doohickey×2
# revenue   = price × quantity  (pre-computed)
SALES_DF = pd.DataFrame({
    'product':  ['Widget', 'Widget', 'Widget', 'Widget',
                 'Gadget', 'Gadget', 'Doohickey', 'Doohickey'],
    'category': ['Elec', 'Elec', 'Elec', 'Elec',
                 'Elec', 'Elec', 'Access', 'Access'],
    'region':   ['North', 'South', 'East', 'West',
                 'North', 'East', 'North', 'South'],
    'price':    [25.0, 25.0, 25.0, 25.0, 150.0, 150.0, 8.0, 8.0],
    'quantity': [10, 5, 4, 6, 3, 7, 50, 15],
    'revenue':  [250.0, 125.0, 100.0, 150.0, 450.0, 1050.0, 400.0, 120.0],
})

import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }

import pandas as pd

def top_groups(df: pd.DataFrame, group_col: str, value_col: str,
               n: int = 5) -> pd.DataFrame:
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )

## Your Implementation

In [ ]:
def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Correlate all numeric columns with target_col.

    Returns DataFrame with columns ['feature', 'correlation'],
    sorted by absolute correlation descending, index reset.
    target_col itself is excluded from the results.
    """
    # TODO: corr = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    # TODO: result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    # TODO: add a temporary '_abs' column, sort descending, drop '_abs'
    # TODO: return result.reset_index(drop=True)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns DataFrame
    try:
        assert 'correlation_summary' in globals()
        result = correlation_summary(SALES_DF, 'revenue')
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: correct columns ['feature', 'correlation']
    try:
        result = correlation_summary(SALES_DF, 'revenue')
        assert list(result.columns) == ['feature', 'correlation'], \
            f'columns={list(result.columns)}, expected [feature, correlation]'
        passed += 1; print("\u2705 Check 2: columns=['feature', 'correlation']")
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: target_col not in features
    try:
        result = correlation_summary(SALES_DF, 'revenue')
        assert 'revenue' not in result['feature'].values, \
            "'revenue' should not appear in feature column (it's the target)"
        # numeric cols: price, quantity, revenue → after drop(revenue) → price, quantity
        assert len(result) == 2, \
            f'expected 2 features (price, quantity), got {len(result)}'
        passed += 1; print('\u2705 Check 3: revenue excluded; 2 features returned')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: all correlation values are in [-1, 1]
    try:
        result = correlation_summary(SALES_DF, 'revenue')
        for _, row in result.iterrows():
            c = row['correlation']
            assert -1.0 <= c <= 1.0, \
                f'correlation {c} for {row["feature"]!r} out of [-1, 1]'
        passed += 1; print('\u2705 Check 4: all correlations in [-1.0, 1.0]')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: sorted by absolute value descending
    try:
        result = correlation_summary(SALES_DF, 'revenue')
        abs_vals = result['correlation'].abs().tolist()
        assert abs_vals == sorted(abs_vals, reverse=True), \
            f'not sorted by abs correlation: {abs_vals}'
        passed += 1; print(f'\u2705 Check 5: sorted by |correlation| desc: {[round(v,3) for v in abs_vals]}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    corr   = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)
```

</details>